# Vocabulary Extension Strategy and Documentation

Given a llama 3 base tokenizer pretrained mainly on high resource langauges we want to extned the vocabulary of the tokeniezr so taht its appropriate for our huge multilingual corpus

However we dont want to extend the vocab in every langauge. E.g we dont need to add english vocabs since the tokenizer is already wenn suited for english data.

Therefore we first identify which languges we need to extend the tokenizer for.
### Step 1: Detect New (Unknown) Languages
We test how well the base tokenizer handles well the base tokenizer handles all the languages.

We run the tokenizer on a sample of each language and collect:
- **Characters per token**
- **Pieces per word**
- **Unknown token rate**

Languages showing high fragmentation or many unknown tokens are considered *new or poorly covered* and become candidates for vocabulary extension.
For this step the script compute_tokenizer_coverage.py was written, it extracts the output in .metric/tokenizer_metric.csvimport pandas as pd




In [3]:
import pandas as pd
df = pd.read_csv("/scratch/hpc-prf-merlin/moe-study/data_prep/metrics/tokenizer_metrics_llama3.2-1b_all_.csv")
print("Best covered languages:")
print(df.nsmallest(15, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

print("\n Worst covered languages:")
print(df.nlargest(5, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

Best covered languages:
     language  pieces_per_word  chars_per_token  unknown_rate
197   english         1.312877         4.517402           0.0
195  spa_Latn         1.619902         3.740534           0.0
191  fas_Arab         1.658605         3.029692           0.0
100  nin_Latn         1.709528         2.516823           0.0
54   ile_Latn         1.717754         3.425544           0.0
78   lbb_Latn         1.782787         2.691116           0.0
32   csy_Latn         1.807866         2.718838           0.0
161  hin_Latn         1.815506         2.886623           0.0
139  xmm_Latn         1.866019         2.983659           0.0
106  jbo_Latn         1.878802         2.577222           0.0
115  nii_Latn         1.918645         2.618079           0.0
26   ibg_Latn         1.997961         3.040667           0.0
126  whg_Latn         2.000003         2.813174           0.0
141  mww_Latn         2.005804         2.474800           0.0
77   lgl_Latn         2.008272         2.47729

The unknown token analysis failed, since llama3 tokenizer doesnt have a unknown token:

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
print("unk_token:", tokenizer.unk_token)
print("unk_token_id:", tokenizer.unk_token_id)


/scratch/hpc-prf-merlin/htyllm-pg/joel/conda/envs/moe-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


unk_token: None
unk_token_id: None


I also tried to include multilingual evaluation benchmarks such as Belebele and HellaSwag. However, I decided not to, since many languages were not covered, the evaluation process was very time-consuming, and the language mapping (e.g., eng_Latn → English) between datasets required significant manual work. Additionally, the LM evaluation framework didn’t support some of the languages I wanted — for example, Belebele only provided 31 languages instead of 122.

### Extend of vocabulary extension for each language?
We aim to expand the LLaMA-3 tokenizer from its original 128K vocabulary to 250K tokens, resulting in an additional 122K subword units.
Rather than distributing these new tokens uniformly, we allocate them proportionally to each language’s tokenizer performance.
Languages that exhibit higher fragmentation or lower coverage (i.e., where the base tokenizer performs poorly) receive a larger share of new tokens, while well-covered languages receive fewer or none.
This adaptive allocation ensures that the extended vocabulary grows efficiently and improves coverage where it is most needed.

### Vocabulary extension ranking-based
We expand LLaMA-3 from **128K → 250K** (+**122K** tokens).
Each language gets tokens **proportional to a continuous “inefficiency” score** derived from:
- higher = worse: *pieces per word* (PPW)
- lower = worse: *characters per token* (CPT)
- (optional) *unknown rate* (UNK; here mostly 0 so weight = 0)

We emphasize under-covered languages by raising the score to a power γ>1.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

TOTAL_NEW_TOKENS = 122_000
W_PPW, W_CPT, W_UNK = 0.5, 0.5, 0.0   # weights for each metric, Note: unknown token id does not exist for out tokenizer
GAMMA = 2.0                            # nonlinearity (γ > 1 emphasizes bad langs)
CSV_PATH = "/scratch/hpc-prf-merlin/moe-study/data_prep/metrics/tokenizer_metrics_llama3.2-1b_all_.csv"
df = pd.read_csv(CSV_PATH)
df.head()


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# normalize metrics
scaler = MinMaxScaler()
dfn = df.copy()
dfn["ppw_norm"] = scaler.fit_transform(dfn[["pieces_per_word"]])
dfn["cpt_norm"] = 1 - scaler.fit_transform(dfn[["chars_per_token"]])
dfn["unk_norm"] = scaler.fit_transform(dfn[["unknown_rate"]])
dfn[["language", "ppw_norm", "cpt_norm", "unk_norm"]].head()


In [ ]:
# linear inefficiency score that indicated how inefficient the tokenizer is in a specific language
dfn["I_linear"] = (
    W_PPW * dfn["ppw_norm"] +
    W_CPT * dfn["cpt_norm"] +
    W_UNK * dfn["unk_norm"]
)

print("linear inefficiency scores: ")
dfn[["language", "I_linear"]].head()


In [ ]:
# Apply nonlinear scaling to emphasize under-covered languages
dfn["I_nonlinear"] = np.power(dfn["I_linear"], GAMMA)
dfn["inefficiency_score"] = MinMaxScaler().fit_transform(dfn[["I_nonlinear"]])

print(f"Applied nonlinear scaling (gamma = {GAMMA}).")
dfn = dfn.sort_values("inefficiency_score", ascending=False)
dfn[["language", "pieces_per_word", "chars_per_token", "inefficiency_score"]].head(10)


In [ ]:
#  now we allocate total new tokens proportionally to inefficiency scores
weights = dfn["inefficiency_score"].to_numpy()
total_w = weights.sum()

if total_w == 0:
    dfn["token_alloc"] = 0
else:
    ideal = weights / total_w * TOTAL_NEW_TOKENS
    floor = np.floor(ideal).astype(int)
    remainder = ideal - floor
    shortfall = TOTAL_NEW_TOKENS - floor.sum()
    if shortfall > 0:
        idx = np.argsort(-remainder)[:shortfall]
        floor[idx] += 1
    dfn["token_alloc"] = floor

assert dfn["token_alloc"].sum() == TOTAL_NEW_TOKENS, "Check allocation sum!"
print(f"Sum of allocated tokens: {dfn['token_alloc'].sum()} (target = {TOTAL_NEW_TOKENS})")


In [ ]:
print("\nbest 15 languages with token allocation:")
print(dfn[["language", "inefficiency_score", "token_alloc"]].head(15))

print("\nworst 10 (best-covered) languages:")
print(dfn[["language", "inefficiency_score", "token_alloc"]].tail(10))


In [ ]:
dfn.to_csv("metrics/vocab_extension_plan_ranking_based_all_llama_3.2-1b.csv", index=False)
print("saved")